In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    current_timestamp,
    input_file_name,
    lit,
    to_date,
)

In [0]:
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

In [0]:
spark = SparkSession.builder.getOrCreate()
dbutils.widgets.removeAll()
dbutils.widgets.text("storage-account", "adlsmartdata1707")
storage_account = dbutils.widgets.get("storage-account")

dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("process_date", "2026-07-17")
dbutils.widgets.text("execution_id", "")

In [0]:
environment = dbutils.widgets.get("environment")
process_date = dbutils.widgets.get("process_date")
execution_id = dbutils.widgets.get("execution_id")

catalog = f"instacart_{environment}"

In [0]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("user_id", IntegerType(), False),
    StructField("eval_set", StringType(), False),
    StructField("order_number", IntegerType(), False),
    StructField("order_dow", IntegerType(), False),
    StructField("order_hour_of_day", IntegerType(), False),
    StructField("days_since_prior_order", DoubleType(), True),
])

In [0]:
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    f"")

In [0]:
raw_path = (
    f"abfss://raw@{storage_account}.dfs.core.windows.net/orders/orders.csv"
)

In [0]:
orders_df = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .schema(orders_schema)
    .csv(raw_path)
)

In [0]:
from pyspark.sql.functions import to_date, when, col

bronze_orders_df = orders_df.select(
    "*",
    when(lit(process_date) == "", "2026-07-17")
    .otherwise(to_date(lit(process_date), "2026-07-17"))
    .alias("process_date"),

    current_timestamp().alias("ingestion_timestamp"),
    input_file_name().alias("source_file"),
    lit("INSTACART").alias("source_system"),
    lit(execution_id).alias("execution_id"),
)

In [0]:
(
    bronze_orders_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.bronze.orders")
)